In [1]:
import numpy as np
from PIL import Image
# import torch
# from torch.utils.data import Dataset
import random
from typing import Tuple, List


In [2]:

# 1) Load image and convert to 0-1 binary mask
# Orange (Netherlands) vs Purple (Belgium) — we map: Netherlands->0, Belgium->1
img_path = "/home/karthik/assignment-3-karthikvmala-main/dataset/Q1/border.png"
img = Image.open(img_path).convert("RGB")
arr = np.array(img)  # shape (H, W, 3)

# Identify the two colors. We'll classify by which channel dominates (R for orange, B for purple)
# orange ~ high R, low B; purple ~ high B, low R
red = arr[..., 0].astype(np.int32)
blue = arr[..., 2].astype(np.int32)

# Label 1 for Belgium (purple -> blue >= red), 0 for Netherlands (orange -> red > blue)
mask = (blue >= red).astype(np.uint8)  # shape (H, W), values in {0,1}

H, W = mask.shape
print("Mask shape:", mask.shape, "| unique labels:", np.unique(mask))

Mask shape: (50, 50) | unique labels: [0 1]


In [ ]:
class PixelBorderDataset:
    """Dataset that returns shuffled, normalized pixel coordinates and labels.

    Each sample: ((x, y), L), where x,y in [0,1]. L in {0,1}.
    Coordinates use x as column index, y as row index.
    """
    def __init__(self, mask: np.ndarray):
        assert mask.ndim == 2, "mask must be HxW"
        self.height, self.width = mask.shape
        # Prepare all coordinates and labels
        ys, xs = np.meshgrid(np.arange(self.height), np.arange(self.width), indexing="ij")
        xs = xs.reshape(-1)
        ys = ys.reshape(-1)
        labels = mask.reshape(-1)
        # Normalize to [0,1]
        xs_norm = xs.astype(np.float32) / (self.width - 1)
        ys_norm = ys.astype(np.float32) / (self.height - 1)
        # Pack
        self.samples: List[Tuple[Tuple[float, float], int]] = [
            ((x, y), int(l)) for x, y, l in zip(xs_norm, ys_norm, labels)
        ]
        # Shuffle once per epoch trigger via random order accessor
        self._order = list(range(len(self.samples)))

    def __len__(self) -> int:
        return len(self.samples)

    def shuffle(self):
        random.shuffle(self._order)

    def __getitem__(self, idx: int):
        sample_idx = self._order[idx]
        (x, y), l = self.samples[sample_idx]
        return ((x, y), l)


In [7]:
# 2) Build dataset and show a few samples

dataset = PixelBorderDataset(mask)
# Shuffle once before first epoch
dataset.shuffle()

# Inspect a few samples
for i in range(5):
    xy, label = dataset[i]
    print(f"sample {i}: (x,y)={xy}, L={int(label)}")

# Optional: create a DataLoader-like utility that reshuffles every epoch
def custom_dataloader(dataset, batch_size, shuffle=True):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)
    for start in range(0, len(dataset), batch_size):
        end = min(start + batch_size, len(dataset))
        batch = [dataset[idx] for idx in indices[start:end]]
        xs = [b[0] for b in batch]
        ys = [b[1] for b in batch]
        yield xs, ys

# Example usage of the custom dataloader
for batch_x, batch_y in custom_dataloader(dataset, batch_size=64):
    print("Batch X:", batch_x)
    print("Batch Y:", batch_y)
    break  # Only showing the first batch


sample 0: (x,y)=(np.float32(0.97959185), np.float32(0.24489796)), L=0
sample 1: (x,y)=(np.float32(0.7755102), np.float32(0.6530612)), L=0
sample 2: (x,y)=(np.float32(0.85714287), np.float32(0.3877551)), L=0
sample 3: (x,y)=(np.float32(0.020408163), np.float32(0.020408163)), L=0
sample 4: (x,y)=(np.float32(1.0), np.float32(0.42857143)), L=0
Batch X: [(np.float32(0.36734694), np.float32(0.53061223)), (np.float32(0.7755102), np.float32(0.3265306)), (np.float32(0.20408164), np.float32(0.9183673)), (np.float32(0.8367347), np.float32(0.85714287)), (np.float32(0.81632656), np.float32(0.97959185)), (np.float32(0.59183675), np.float32(0.20408164)), (np.float32(0.59183675), np.float32(0.81632656)), (np.float32(0.3469388), np.float32(0.30612245)), (np.float32(0.6938776), np.float32(0.3469388)), (np.float32(0.0), np.float32(1.0)), (np.float32(0.7346939), np.float32(0.877551)), (np.float32(0.9591837), np.float32(0.040816326)), (np.float32(0.6938776), np.float32(0.020408163)), (np.float32(1.0), np.f

In [8]:
# Activation function classes (NumPy, from scratch)
import numpy as _np

class Activation:
    def forward(self, x: _np.ndarray) -> _np.ndarray:
        raise NotImplementedError
    def backward(self, grad_output: _np.ndarray) -> _np.ndarray:
        raise NotImplementedError

class Identity(Activation):
    def __init__(self):
        self.last_input = None
    def forward(self, x: _np.ndarray) -> _np.ndarray:
        self.last_input = x
        return x
    def backward(self, grad_output: _np.ndarray) -> _np.ndarray:
        return grad_output

class ReLU(Activation):
    def __init__(self):
        self.last_input = None
    def forward(self, x: _np.ndarray) -> _np.ndarray:
        self.last_input = x
        return _np.maximum(0.0, x)
    def backward(self, grad_output: _np.ndarray) -> _np.ndarray:
        mask = (self.last_input > 0).astype(grad_output.dtype)
        return grad_output * mask

class Sigmoid(Activation):
    def __init__(self):
        self.last_output = None
    def forward(self, x: _np.ndarray) -> _np.ndarray:
        # Numerically stable sigmoid
        out = 1.0 / (1.0 + _np.exp(-x))
        self.last_output = out
        return out
    def backward(self, grad_output: _np.ndarray) -> _np.ndarray:
        s = self.last_output
        return grad_output * s * (1.0 - s)

class Tanh(Activation):
    def __init__(self):
        self.last_output = None
    def forward(self, x: _np.ndarray) -> _np.ndarray:
        out = _np.tanh(x)
        self.last_output = out
        return out
    def backward(self, grad_output: _np.ndarray) -> _np.ndarray:
        t = self.last_output
        return grad_output * (1.0 - t**2)



In [9]:
# Linear layer (NumPy) with stored forward pass data and accumulated grads
class Linear:
    def __init__(self, in_features: int, out_features: int, activation: Activation):
        self.in_features = in_features
        self.out_features = out_features
        self.activation = activation
        # Kaiming/He uniform for ReLU by default; simple uniform otherwise
        limit = _np.sqrt(6.0 / (in_features + out_features))
        self.W = _np.random.uniform(-limit, limit, size=(out_features, in_features)).astype(_np.float32)
        self.b = _np.zeros((out_features,), dtype=_np.float32)
        # caches for forward/backward
        self.last_input = None           # pre-activation input to linear
        self.last_linear_output = None   # Wx + b
        self.last_activation_output = None
        # accumulated gradients
        self.dW = _np.zeros_like(self.W)
        self.db = _np.zeros_like(self.b)

    def forward(self, x: _np.ndarray) -> _np.ndarray:
        # x: (N, in_features)
        self.last_input = x
        lin = x @ self.W.T + self.b  # (N, out_features)
        self.last_linear_output = lin
        act = self.activation.forward(lin)
        self.last_activation_output = act
        return act

    def backward(self, grad_output: _np.ndarray) -> _np.ndarray:
        # grad_output: dL/dy where y is post-activation output
        grad_act_in = self.activation.backward(grad_output)  # shape (N, out_features)
        # grads w.r.t. parameters
        # dL/dW = grad_act_in^T @ x
        self.dW += grad_act_in.T @ self.last_input
        # dL/db = sum over batch
        self.db += grad_act_in.sum(axis=0)
        # grad w.r.t. input: grad_act_in @ W
        grad_input = grad_act_in @ self.W
        return grad_input

    def zero_grad(self):
        self.dW.fill(0.0)
        self.db.fill(0.0)

    def step(self, lr: float):
        self.W -= lr * self.dW
        self.b -= lr * self.db




In [10]:
# Model class with MSE and BCE losses
class Model:
    def __init__(self, layers: list, loss: str = "mse", lr: float = 1e-2):
        assert len(layers) > 0, "Model requires at least one layer"
        self.layers = layers
        self.lr = lr
        loss = loss.lower()
        assert loss in ("mse", "bce"), "loss must be 'mse' or 'bce'"
        self.loss_type = loss

    # ----- Losses and their gradients -----
    def _mse_loss(self, y_pred: _np.ndarray, y_true: _np.ndarray):
        # mean over batch and features
        diff = y_pred - y_true
        loss = _np.mean(diff ** 2)
        grad = (2.0 / y_pred.size) * diff
        return loss, grad

    def _bce_loss(self, y_pred: _np.ndarray, y_true: _np.ndarray, eps: float = 1e-12):
        # y_pred assumed in (0,1); add clamping for stability
        y_pred = _np.clip(y_pred, eps, 1.0 - eps)
        loss = -_np.mean(y_true * _np.log(y_pred) + (1.0 - y_true) * _np.log(1.0 - y_pred))
        # gradient of mean BCE wrt y_pred
        grad = (y_pred - y_true) / (y_pred * (1.0 - y_pred))
        grad /= y_pred.shape[0] * y_pred.shape[1] if y_pred.ndim == 2 else y_pred.size
        return loss, grad

    def _loss_and_grad(self, y_pred: _np.ndarray, y_true: _np.ndarray):
        if self.loss_type == "mse":
            return self._mse_loss(y_pred, y_true)
        else:
            return self._bce_loss(y_pred, y_true)

    # ----- Core passes -----
    def forward(self, x: _np.ndarray) -> _np.ndarray:
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, grad_output: _np.ndarray):
        grad = grad_output
        # traverse layers in reverse
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    # ----- Training helpers -----
    def train(self, x: _np.ndarray, y: _np.ndarray):
        # forward
        y_pred = self.forward(x)
        # loss + dL/dy_pred
        loss, grad = self._loss_and_grad(y_pred, y)
        # backward accumulates grads in layers
        self.backward(grad)
        return loss, y_pred

    def zero_grad(self):
        for layer in self.layers:
            layer.zero_grad()

    def update(self):
        for layer in self.layers:
            layer.step(self.lr)
        self.zero_grad()

    def predict(self, x: _np.ndarray) -> _np.ndarray:
        return self.forward(x)

    def save_to(self, path: str):
        # save all weights and biases with layer index
        data = {}
        for i, layer in enumerate(self.layers):
            data[f"W_{i}"] = layer.W
            data[f"b_{i}"] = layer.b
        _np.savez(path, **data)

    def load_from(self, path: str):
        loaded = _np.load(path)
        # check shapes and copy
        for i, layer in enumerate(self.layers):
            W_key, B_key = f"W_{i}", f"b_{i}"
            assert W_key in loaded and B_key in loaded, f"Missing {W_key}/{B_key} in file"
            W_arr, B_arr = loaded[W_key], loaded[B_key]
            if W_arr.shape != layer.W.shape or B_arr.shape != layer.b.shape:
                raise ValueError(
                    f"Shape mismatch at layer {i}: expected W{layer.W.shape}, b{layer.b.shape}, "
                    f"got W{W_arr.shape}, b{B_arr.shape}"
                )
            layer.W[...] = W_arr
            layer.b[...] = B_arr
            
    def forward_and_loss(self, x: _np.ndarray, y: _np.ndarray) -> tuple:
        """
        Forward pass only (no backprop) to compute loss for gradient checking.
        """
        y_pred = self.forward(x)
        loss, _ = self._loss_and_grad(y_pred, y)
        return loss, y_pred



In [11]:
def gradient_check(model, X, Y, epsilon=1e-5, atol=1e-4):
    # First, compute analytical gradients via backprop
    model.zero_grad()
    loss, _ = model.train(X, Y)  # This should compute loss and populate model grads

    # Collect analytical gradients
    analytical_grads = []
    for layer in model.layers:
        if hasattr(layer, 'W'):
            analytical_grads.append(layer.dW.copy())  # dW: gradient from backprop

    # Now compute numerical gradients
    numerical_grads = []
    for layer in model.layers:
        if not hasattr(layer, 'W'):
            continue  # Skip layers without weights

        W = layer.W
        dW_numeric = _np.zeros_like(W)

        for i in range(W.shape[0]):
            for j in range(W.shape[1]):
                old_val = W[i, j]

                # Compute f(theta + epsilon)
                W[i, j] = old_val + epsilon
                loss_plus, _ = model.forward_and_loss(X, Y)

                # Compute f(theta - epsilon)
                W[i, j] = old_val - epsilon
                loss_minus, _ = model.forward_and_loss(X, Y)

                # Numerical gradient
                dW_numeric[i, j] = (loss_plus - loss_minus) / (2 * epsilon)

                # Reset the weight
                W[i, j] = old_val

        numerical_grads.append(dW_numeric)

    # Compare analytical and numerical gradients
    for idx, (grad_an, grad_num) in enumerate(zip(analytical_grads, numerical_grads)):
        print(f"Layer {idx} gradient check.")
        diff = _np.abs(grad_an - grad_num)
        print("Max difference:", diff.max())
        


In [12]:
import numpy as _np

# Assume these are defined elsewhere
# from your_module import Model, Linear, ReLU, Sigmoid

# XOR-like dataset (not linearly separable)
X = _np.array([[0., 0.],
               [0., 1.],
               [1., 0.],
               [1., 1.]], dtype=_np.float32)

Y = _np.array([[0.],
               [1.],
               [1.],
               [0.]], dtype=_np.float32)

# Set random seed for reproducibility (optional but useful)
_np.random.seed(42)

# Define model
net = Model([
    Linear(2, 8, ReLU()),
    Linear(8, 1, Sigmoid()),
], loss="bce", lr=0.1)

# Training loop
for epoch in range(2000):
    net.zero_grad()
    loss, _ = net.train(X, Y)
    net.update()
    if (epoch + 1) % 400 == 0:
        print(f"Epoch {epoch + 1}, Loss = {float(loss):.4f}")

# Predictions after training
preds = net.predict(X)
print("\nPredictions after training (rounded):\n", preds.round(3))
print("Predicted labels:\n", preds.round())
print("True labels:\n", Y.T)

# Save & load check
path = "/home/karthik/assignment-3-karthikvmala-main/tmp_model.npz"
net.save_to(path)

# Load into new model instance
net2 = Model([
    Linear(2, 8, ReLU()),
    Linear(8, 1, Sigmoid()),
], loss="bce", lr=0.1)
net2.load_from(path)

# Confirm predictions match

preds_loaded = net2.predict(X)
print("\nLoaded model predictions match original:", _np.allclose(preds, preds_loaded, atol=1e-6))
print("Loaded model predictions (rounded):\n", preds_loaded.round(3))
gradient_check(net, X, Y)

Epoch 400, Loss = 0.1999
Epoch 800, Loss = 0.0446
Epoch 1200, Loss = 0.0226
Epoch 1600, Loss = 0.0147
Epoch 2000, Loss = 0.0108

Predictions after training (rounded):
 [[0.03 ]
 [0.996]
 [0.995]
 [0.004]]
Predicted labels:
 [[0.]
 [1.]
 [1.]
 [0.]]
True labels:
 [[0. 1. 1. 0.]]

Loaded model predictions match original: True
Loaded model predictions (rounded):
 [[0.03 ]
 [0.996]
 [0.995]
 [0.004]]
Layer 0 gradient check.
Max difference: 0.00067091826
Layer 1 gradient check.
Max difference: 0.00073755905


In [13]:
# Training utilities: batching, grad accumulation, logging, early stopping, plotting, wandb
import time, os
from datetime import datetime
import matplotlib.pyplot as plt
from tqdm import trange

try:
    import wandb
    _WANDB = True
except Exception:
    _WANDB = False


def iterate_minibatches(X: _np.ndarray, Y: _np.ndarray, batch_size: int, shuffle: bool = True):
    N = X.shape[0]
    idx = _np.arange(N)
    if shuffle:
        _np.random.shuffle(idx)
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch_idx = idx[start:end]
        yield X[batch_idx], Y[batch_idx]


def train_model(
    model: Model,
    X: _np.ndarray,
    Y: _np.ndarray,
    batch_size: int = 128,
    grad_accum_steps: int = 1,
    epochs: int = 100,
    patience: int = 10,
    rel_improve_threshold: float = 0.01,
    run_name: str = None,
    use_wandb: bool = True,
):
    assert grad_accum_steps >= 1
    assert batch_size >= 1

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") if run_name is None else run_name
    run_dir = f"/home/karthik/assignment-3-karthikvmala-main/runs/{timestamp}"
    os.makedirs(run_dir, exist_ok=True)

    if use_wandb and _WANDB:
        wandb.init(project="mlp-border", name=timestamp, config={
            "batch_size": batch_size,
            "grad_accum_steps": grad_accum_steps,
            "epochs": epochs,
            "lr": model.lr,
            "loss": model.loss_type,
            "patience": patience,
            "rel_improve_threshold": rel_improve_threshold,
        })

    samples_seen = 0
    losses = []
    epoch_losses = []

    for epoch in trange(epochs, desc="epochs"):
        model.zero_grad()
        accum_counter = 0
        running_loss = 0.0
        num_batches = 0
        for xb, yb in iterate_minibatches(X, Y, batch_size, shuffle=True):
            loss, _ = model.train(xb, yb)
            running_loss += float(loss)
            num_batches += 1
            accum_counter += 1
            samples_seen += xb.shape[0]
            losses.append((samples_seen, float(loss)))
            if use_wandb and _WANDB:
                wandb.log({"loss": float(loss), "samples_seen": samples_seen, "epoch": epoch})
            if accum_counter >= grad_accum_steps:
                model.update()
                accum_counter = 0
        # flush any leftovers
        if accum_counter > 0:
            model.update()
        avg_epoch_loss = running_loss / max(1, num_batches)
        epoch_losses.append(avg_epoch_loss)
        if use_wandb and _WANDB:
            wandb.log({"epoch_loss": avg_epoch_loss, "epoch": epoch})

        # Early stopping after enough epochs
        if len(epoch_losses) > patience:
            L_t = epoch_losses[-1]
            L_tm10 = epoch_losses[-1 - patience]
            if L_t >= (1.0 - rel_improve_threshold) * L_tm10:
                print(f"Early stopping at epoch {epoch} (loss {L_t:.6f} vs {L_tm10:.6f})")
                break

    # Plot loss vs samples_seen
    if len(losses) > 0:
        xs, ys = zip(*losses)
        plt.figure(figsize=(6,4))
        plt.plot(xs, ys, label="train loss")
        plt.xlabel("samples seen")
        plt.ylabel("loss")
        plt.title("Training loss vs samples")
        plt.legend()
        fig_path = os.path.join(run_dir, "loss_vs_samples.png")
        plt.tight_layout()
        plt.savefig(fig_path, dpi=150)
        plt.close()
        if use_wandb and _WANDB:
            wandb.log({"loss_plot": wandb.Image(fig_path)})

    # Save final model parameters
    weights_path = os.path.join(run_dir, "model_final.npz")
    model.save_to(weights_path)

    # Save run metadata
    meta = {
        "batch_size": batch_size,
        "grad_accum_steps": grad_accum_steps,
        "epochs": epochs,
        "lr": model.lr,
        "loss": model.loss_type,
        "patience": patience,
        "rel_improve_threshold": rel_improve_threshold,
        "final_epoch": len(epoch_losses)-1,
        "final_loss": float(epoch_losses[-1]) if epoch_losses else None,
    }
    _np.savez(os.path.join(run_dir, "run_meta.npz"), **{k: _np.array(v) for k,v in meta.items()})

    if use_wandb and _WANDB:
        wandb.summary["final_loss"] = meta["final_loss"]
        wandb.finish()

    return {
        "run_dir": run_dir,
        "losses": losses,
        "epoch_losses": epoch_losses,
        "samples_seen": samples_seen,
    }



In [14]:
# Demo: train on pixel border dataset with BCE
# Build full dataset arrays X (N,2), Y (N,1)

# Reuse mask from earlier cell
coords = []
labels = []
for y in range(H):
    for x in range(W):
        coords.append([x / (W - 1), y / (H - 1)])
        labels.append([mask[y, x]])
X_all = _np.array(coords, dtype=_np.float32)
Y_all = _np.array(labels, dtype=_np.float32)

mlp = Model([
    Linear(2, 32, ReLU()),
    Linear(32, 32, ReLU()),
    Linear(32, 1, Sigmoid()),
], loss="bce", lr=0.05)

result = train_model(
    mlp,
    X_all,
    Y_all,
    batch_size=256,
    grad_accum_steps=2,
    epochs=200,
    patience=10,
    rel_improve_threshold=0.01,
    use_wandb=False,
)

print("Run dir:", result["run_dir"], "samples_seen:", result["samples_seen"], "final epoch loss:", result["epoch_losses"][-1])


epochs:  10%|█         | 21/200 [00:00<00:00, 181.37it/s]


Early stopping at epoch 21 (loss 0.508816 vs 0.511549)
Run dir: /home/karthik/assignment-3-karthikvmala-main/runs/20251013_135918 samples_seen: 55000 final epoch loss: 0.5088158249855042


In [15]:
# XOR dataset and helpers
X_xor = _np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=_np.float32)
Y_xor = _np.array([[0.],[1.],[1.],[0.]], dtype=_np.float32)

def accuracy(pred: _np.ndarray, y: _np.ndarray) -> float:
    if pred.ndim == 2 and pred.shape[1] == 1:
        pred_bin = (pred >= 0.5).astype(_np.float32)
    else:
        pred_bin = (pred >= 0.5).astype(_np.float32)
    return float((pred_bin == y).mean())


def run_xor(activation_hidden, hidden_width: int, depth: int, loss: str = "bce", lr: float = 0.1, epochs: int = 5000):
    layers = []
    in_dim = 2
    out_dim = 1
    # first hidden
    layers.append(Linear(in_dim, hidden_width, activation_hidden()))
    # additional hidden layers
    for _ in range(max(0, depth - 1)):
        layers.append(Linear(hidden_width, hidden_width, activation_hidden()))
    # output with Sigmoid for BCE
    layers.append(Linear(hidden_width, out_dim, Sigmoid() if loss == "bce" else Identity()))
    model = Model(layers, loss=loss, lr=lr)

    for ep in range(epochs):
        model.zero_grad()
        loss_val, _ = model.train(X_xor, Y_xor)
        model.update()
        if (ep+1) % 1000 == 0:
            preds = model.predict(X_xor)
            acc = accuracy(preds, Y_xor)
            print(f"{activation_hidden.__name__} depth={depth} width={hidden_width} ep={ep+1} loss={loss_val:.4f} acc={acc:.3f}")
            if acc == 1.0:
                break
    preds = model.predict(X_xor)
    acc = accuracy(preds, Y_xor)
    return model, float(loss_val), acc



In [16]:
# Sweep several architectures and activations; assert 100% accuracy
configs = [
    (ReLU, 4, 1),
    (ReLU, 8, 2),
    (Tanh, 4, 1),
    (Tanh, 8, 2),
    (Sigmoid, 4, 2),
    (Identity, 8, 2),  # Identity alone can't solve XOR; include hidden Identity only if stacked with nonlinearity elsewhere
]

# We'll skip pure Identity-only networks. For Identity, mix with nonlinearity by keeping Sigmoid output.

results = []
for act, width, depth in configs:
    if act is Identity:
        # add a Tanh hidden in the middle to ensure nonlinearity
        layers = [
            Linear(2, width, Identity()),
            Linear(width, width, Tanh()),
            Linear(width, 1, Sigmoid()),
        ]
        model = Model(layers, loss="bce", lr=0.2)
        for ep in range(5000):
            model.zero_grad(); model.train(X_xor, Y_xor); model.update()
        acc = accuracy(model.predict(X_xor), Y_xor)
        print(f"Identity-mixed acc={acc:.3f}")
        assert acc == 1.0
        results.append(("Identity-mixed", width, 2, acc))
    else:
        model, loss_val, acc = run_xor(act, width, depth, loss="bce", lr=0.2, epochs=5000)
        print(f"{act.__name__} width={width} depth={depth} acc={acc}")
        assert acc == 1.0
        results.append((act.__name__, width, depth, acc))

print("All XOR runs converged to 100% accuracy.")


ReLU depth=1 width=4 ep=1000 loss=0.0105 acc=1.000
ReLU width=4 depth=1 acc=1.0
ReLU depth=2 width=8 ep=1000 loss=0.0009 acc=1.000
ReLU width=8 depth=2 acc=1.0
Tanh depth=1 width=4 ep=1000 loss=0.0184 acc=1.000
Tanh width=4 depth=1 acc=1.0
Tanh depth=2 width=8 ep=1000 loss=0.0024 acc=1.000
Tanh width=8 depth=2 acc=1.0
Sigmoid depth=2 width=4 ep=1000 loss=0.6911 acc=0.750
Sigmoid depth=2 width=4 ep=2000 loss=0.6643 acc=0.750
Sigmoid depth=2 width=4 ep=3000 loss=0.2108 acc=1.000
Sigmoid width=4 depth=2 acc=1.0
Identity-mixed acc=1.000
All XOR runs converged to 100% accuracy.


In [17]:
# Numerical gradient checking on a small XOR network

def numerical_grad_check(model: Model, x: _np.ndarray, y: _np.ndarray, eps: float = 1e-4):
    # compute backprop gradients first
    model.zero_grad()
    loss_val, _ = model.train(x, y)
    # do not update; we want grads in-place
    backprop_grads = []
    for layer in model.layers:
        backprop_grads.append((layer.dW.copy(), layer.db.copy()))

    # numerical grads via central difference on each parameter
    num_grads = []
    for li, layer in enumerate(model.layers):
        gW = _np.zeros_like(layer.W)
        gb = _np.zeros_like(layer.b)
        # weights
        for i in range(layer.W.shape[0]):
            for j in range(layer.W.shape[1]):
                orig = layer.W[i, j]
                layer.W[i, j] = orig + eps
                lp, _ = model._loss_and_grad(model.predict(x), y)
                layer.W[i, j] = orig - eps
                lm, _ = model._loss_and_grad(model.predict(x), y)
                layer.W[i, j] = orig
                gW[i, j] = (lp - lm) / (2.0 * eps)
        # biases
        for i in range(layer.b.shape[0]):
            orig = layer.b[i]
            layer.b[i] = orig + eps
            lp, _ = model._loss_and_grad(model.predict(x), y)
            layer.b[i] = orig - eps
            lm, _ = model._loss_and_grad(model.predict(x), y)
            layer.b[i] = orig
            gb[i] = (lp - lm) / (2.0 * eps)
        num_grads.append((gW, gb))
    return backprop_grads, num_grads

# Build a tiny model for checking
check_model = Model([
    Linear(2, 3, Tanh()),
    Linear(3, 1, Sigmoid()),
], loss="bce", lr=0.1)

bp, num = numerical_grad_check(check_model, X_xor, Y_xor, eps=1e-5)

# Compute relative error per layer
for li, ((dW_bp, db_bp), (dW_num, db_num)) in enumerate(zip(bp, num)):
    rel_W = _np.linalg.norm(dW_bp - dW_num) / (1e-12 + _np.linalg.norm(dW_bp) + _np.linalg.norm(dW_num))
    rel_b = _np.linalg.norm(db_bp - db_num) / (1e-12 + _np.linalg.norm(db_bp) + _np.linalg.norm(db_num))
    print(f"Layer {li}: rel_err W={rel_W:.6e}, b={rel_b:.6e}")



Layer 0: rel_err W=2.574684e-02, b=2.228890e-02
Layer 1: rel_err W=1.671665e-02, b=1.268412e-02


In [18]:


def preds_to_map(model: Model, H: int, W: int) -> _np.ndarray:
    coords = _np.stack(_np.meshgrid(_np.arange(W), _np.arange(H), indexing="xy"), axis=-1).reshape(-1,2)
    coords = coords.astype(_np.float32)
    coords[:,0] /= (W - 1)
    coords[:,1] /= (H - 1)
    pred = model.predict(coords).reshape(H, W, -1)
    if pred.shape[2] == 1:
        pred = pred[...,0]
    return (pred >= 0.5).astype(_np.uint8), pred


def map_accuracy(pred_mask: _np.ndarray, gt_mask: _np.ndarray) -> float:
    return float((pred_mask == gt_mask).mean())


def save_triptych(run_dir: str, gt_mask: _np.ndarray, pred_mask: _np.ndarray, err_mask: _np.ndarray, title: str = None):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9,3))
    plt.subplot(1,3,1)
    plt.imshow(gt_mask, cmap="gray")
    plt.title("Ground Truth"); plt.axis("off")
    plt.subplot(1,3,2)
    plt.imshow(pred_mask, cmap="gray")
    plt.title("Prediction"); plt.axis("off")
    plt.subplot(1,3,3)
    # overlay errors in red on GT
    base = gt_mask.copy()
    plt.imshow(base, cmap="gray")
    err_overlay = _np.zeros((gt_mask.shape[0], gt_mask.shape[1], 4), dtype=_np.float32)
    err_overlay[...,0] = err_mask.astype(_np.float32)  # red channel
    err_overlay[...,3] = err_mask.astype(_np.float32) * 0.6  # alpha
    plt.imshow(err_overlay)
    plt.title("Errors (red)"); plt.axis("off")
    if title: plt.suptitle(title)
    out = os.path.join(run_dir, "triptych.png")
    plt.tight_layout(rect=[0,0,1,0.95])
    plt.savefig(out, dpi=150)
    plt.close()
    return out


def train_one_map_run(width: int, depth: int, act_cls, lr: float, batch_size: int, grad_accum_steps: int, epochs: int = 500, use_wandb: bool = False):
    layers = [Linear(2, width, act_cls())]
    for _ in range(depth-1):
        layers.append(Linear(width, width, act_cls()))
    layers.append(Linear(width, 1, Sigmoid()))
    model = Model(layers, loss="bce", lr=lr)

    # data arrays
    coords = []
    labels = []
    for y in range(H):
        for x in range(W):
            coords.append([x/(W-1), y/(H-1)])
            labels.append([mask[y,x]])
    X_arr = _np.array(coords, dtype=_np.float32)
    Y_arr = _np.array(labels, dtype=_np.float32)

    result = train_model(model, X_arr, Y_arr, batch_size=batch_size, grad_accum_steps=grad_accum_steps, epochs=epochs, use_wandb=use_wandb)

    pred_bin, pred_raw = preds_to_map(model, H, W)
    acc = map_accuracy(pred_bin, mask)
    err = (pred_bin != mask).astype(_np.uint8)

    # save images
    trip = save_triptych(result["run_dir"], mask, pred_bin, err, title=f"w{width}-d{depth}-{act_cls.__name__}")
    # append metrics
    meta_path = os.path.join(result["run_dir"], "run_meta.npz")
    meta = dict(_np.load(meta_path))
    meta.update({
        "final_accuracy": _np.array(acc),
        "H": _np.array(H),
        "W": _np.array(W),
        "width": _np.array(width),
        "depth": _np.array(depth),
        "activation": _np.array(act_cls.__name__),
        "triptych_path": _np.array(trip),
    })
    _np.savez(meta_path, **meta)

    return {**result, "accuracy": acc, "triptych": trip}


def sweep_depths(fixed_width: int, act_cls, depths: list, lr: float, batch_size: int, grad_accum_steps: int, epochs: int = 300):
    xs_depth, losses, accs = [], [], []
    for d in depths:
        out = train_one_map_run(fixed_width, d, act_cls, lr, batch_size, grad_accum_steps, epochs, use_wandb=False)
        xs_depth.append(d); losses.append(out["epoch_losses"][-1]); accs.append(out["accuracy"])
    # plot
    plt.figure(figsize=(8,3))
    plt.subplot(1,2,1); plt.plot(xs_depth, losses, marker="o"); plt.xlabel("depth"); plt.ylabel("final loss"); plt.title(f"Loss vs depth (w={fixed_width}, {act_cls.__name__})")
    plt.subplot(1,2,2); plt.plot(xs_depth, accs, marker="o"); plt.xlabel("depth"); plt.ylabel("accuracy"); plt.title(f"Acc vs depth (w={fixed_width}, {act_cls.__name__})")
    out = f"/home/karthik/assignment-3-karthikvmala-main/runs/depth_sweep_w{fixed_width}_{act_cls.__name__}.png"
    plt.tight_layout(); plt.savefig(out, dpi=150); plt.close()
    return out, xs_depth, losses, accs


def sweep_widths(fixed_depth: int, act_cls, widths: list, lr: float, batch_size: int, grad_accum_steps: int, epochs: int = 300):
    xs_width, losses, accs = [], [], []
    for w in widths:
        out = train_one_map_run(w, fixed_depth, act_cls, lr, batch_size, grad_accum_steps, epochs, use_wandb=False)
        xs_width.append(w); losses.append(out["epoch_losses"][-1]); accs.append(out["accuracy"])
    # plot
    plt.figure(figsize=(8,3))
    plt.subplot(1,2,1); plt.plot(xs_width, losses, marker="o"); plt.xlabel("width"); plt.ylabel("final loss"); plt.title(f"Loss vs width (d={fixed_depth}, {act_cls.__name__})")
    plt.subplot(1,2,2); plt.plot(xs_width, accs, marker="o"); plt.xlabel("width"); plt.ylabel("accuracy"); plt.title(f"Acc vs width (d={fixed_depth}, {act_cls.__name__})")
    out = f"/home/karthik/assignment-3-karthikvmala-main/runs/width_sweep_d{fixed_depth}_{act_cls.__name__}.png"
    plt.tight_layout(); plt.savefig(out, dpi=150); plt.close()
    return out, xs_width, losses, accs


def sweep_hyperparams(width: int, depth: int, act_cls, lr_list: list, batch_sizes: list, accum_list: list, epochs: int = 300):
    records = []
    for lr in lr_list:
        for bs in batch_sizes:
            for ga in accum_list:
                start = time.time()
                out = train_one_map_run(width, depth, act_cls, lr, bs, ga, epochs, use_wandb=False)
                elapsed = time.time() - start
                # samples seen equals last samples_seen from train_model
                records.append({
                    "lr": lr, "batch_size": bs, "grad_accum": ga, "final_loss": out["epoch_losses"][-1],
                    "accuracy": out["accuracy"], "time_sec": elapsed, "samples_seen": out["samples_seen"],
                    "run_dir": out["run_dir"],
                })
    # simple table print
    for r in records:
        print(r)
    return records



In [19]:
# Example sweeps (run selectively to avoid long runtimes)
# Single run example
out = train_one_map_run(width=32, depth=2, act_cls=ReLU, lr=0.05, batch_size=256, grad_accum_steps=2, epochs=200, use_wandb=False)
print("Example run:", {k: out[k] for k in ["run_dir","accuracy","samples_seen"]})

# Depth sweep
depth_plot, depths, final_losses, final_accs = sweep_depths(fixed_width=32, act_cls=ReLU, depths=[1,2,3,4], lr=0.05, batch_size=256, grad_accum_steps=2, epochs=200)
print("Depth sweep plot:", depth_plot)

# Width sweep
width_plot, widths, w_losses, w_accs = sweep_widths(fixed_depth=3, act_cls=Tanh, widths=[8,16,32,64], lr=0.05, batch_size=256, grad_accum_steps=2, epochs=200)
print("Width sweep plot:", width_plot)

# Hyperparameter sweep
recs = sweep_hyperparams(width=32, depth=2, act_cls=ReLU, lr_list=[0.02,0.05,0.1], batch_sizes=[128,256], accum_list=[1,2,4], epochs=150)
print("Num runs:", len(recs))


epochs:  11%|█         | 22/200 [00:00<00:00, 485.29it/s]


Early stopping at epoch 22 (loss 0.507379 vs 0.512302)
Example run: {'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_135937', 'accuracy': 0.7944, 'samples_seen': 57500}


epochs:  12%|█▎        | 25/200 [00:00<00:00, 225.48it/s]

Early stopping at epoch 25 (loss 0.501796 vs 0.505523)



epochs:  14%|█▍        | 29/200 [00:00<00:00, 257.58it/s]


...

Early stopping at epoch 40 (loss 0.487900 vs 0.492740)


epochs:  23%|██▎       | 34/150 [00:00<00:00, 306.03it/s]


Early stopping at epoch 34 (loss 0.482684 vs 0.487366)


epochs:  21%|██        | 31/150 [00:00<00:00, 253.64it/s]


Early stopping at epoch 31 (loss 0.488541 vs 0.492243)
{'lr': 0.02, 'batch_size': 128, 'grad_accum': 1, 'final_loss': 0.505981869995594, 'accuracy': 0.7944, 'time_sec': 0.7155616283416748, 'samples_seen': 47500, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_135944'}
{'lr': 0.02, 'batch_size': 128, 'grad_accum': 2, 'final_loss': 0.5011011347174644, 'accuracy': 0.7944, 'time_sec': 0.6484529972076416, 'samples_seen': 57500, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_135945'}
{'lr': 0.02, 'batch_size': 128, 'grad_accum': 4, 'final_loss': 0.5080334231257438, 'accuracy': 0.7944, 'time_sec': 0.5983078479766846, 'samples_seen': 55000, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_135945'}
{'lr': 0.02, 'batch_size': 256, 'grad_accum': 1, 'final_loss': 0.5018702894449234, 'accuracy': 0.7944, 'time_sec': 0.6412405967712402, 'samples_seen': 75000, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_135

In [20]:
# Final challenge utilities: search minimal params and minimal samples for >=91% acc

def count_params(layers: list) -> int:
    total = 0
    for lyr in layers:
        total += lyr.W.size + lyr.b.size
    return int(total)


def build_layers(in_dim: int, out_dim: int, width: int, depth: int, act_cls):
    layers = [Linear(in_dim, width, act_cls())]
    for _ in range(depth-1):
        layers.append(Linear(width, width, act_cls()))
    layers.append(Linear(width, out_dim, Sigmoid()))
    return layers


def search_min_params(target_acc: float = 0.91, widths=(2,4,8,12,16,24,32), depths=(1,2,3,4), act_list=(ReLU, Tanh), lr_list=(0.02,0.05,0.1), epochs=300, batch_size=256, grad_accum_steps=2):
    best = None
    for act in act_list:
        for d in depths:
            for w in widths:
                for lr in lr_list:
                    layers = build_layers(2, 1, w, d, act)
                    params = count_params(layers)
                    model = Model(layers, loss="bce", lr=lr)
                    coords, labels = [], []
                    for y in range(H):
                        for x in range(W):
                            coords.append([x/(W-1), y/(H-1)])
                            labels.append([mask[y,x]])
                    X_arr = _np.array(coords, dtype=_np.float32)
                    Y_arr = _np.array(labels, dtype=_np.float32)
                    out = train_model(model, X_arr, Y_arr, batch_size=batch_size, grad_accum_steps=grad_accum_steps, epochs=epochs, use_wandb=False)
                    acc = map_accuracy(preds_to_map(model, H, W)[0], mask)
                    rec = {
                        "acc": acc, "params": params, "width": w, "depth": d, "act": act.__name__,
                        "lr": lr, "run_dir": out["run_dir"], "final_loss": out["epoch_losses"][-1]
                    }
                    print(rec)
                    if acc >= target_acc:
                        if best is None or params < best["params"] or (params == best["params"] and rec["final_loss"] < best["final_loss"]):
                            best = rec
    return best


def search_min_samples(target_acc: float = 0.91, width: int = 16, depth: int = 2, act_cls=ReLU, lrs=(0.02,0.05,0.1), batch_sizes=(64,128,256), accum_steps=(1,2,4), epochs=300):
    # We measure samples_seen from the training utility
    best = None
    for lr in lrs:
        for bs in batch_sizes:
            for ga in accum_steps:
                layers = build_layers(2, 1, width, depth, act_cls)
                model = Model(layers, loss="bce", lr=lr)
                coords, labels = [], []
                for y in range(H):
                    for x in range(W):
                        coords.append([x/(W-1), y/(H-1)])
                        labels.append([mask[y,x]])
                X_arr = _np.array(coords, dtype=_np.float32)
                Y_arr = _np.array(labels, dtype=_np.float32)
                out = train_model(model, X_arr, Y_arr, batch_size=bs, grad_accum_steps=ga, epochs=epochs, use_wandb=False)
                acc = map_accuracy(preds_to_map(model, H, W)[0], mask)
                rec = {
                    "acc": acc, "samples_seen": out["samples_seen"], "lr": lr, "batch_size": bs, "grad_accum": ga,
                    "run_dir": out["run_dir"], "final_loss": out["epoch_losses"][-1]
                }
                print(rec)
                if acc >= target_acc:
                    if best is None or rec["samples_seen"] < best["samples_seen"] or (rec["samples_seen"] == best["samples_seen"] and rec["final_loss"] < best["final_loss"]):
                        best = rec
    return best



In [21]:
# Example usage: final challenge searches (may take a while)
best_params = search_min_params(target_acc=0.91, widths=(2,4,8,12,16,24,32), depths=(1,2,3,4), act_list=(ReLU,Tanh), lr_list=(0.02,0.05,0.1), epochs=250, batch_size=256, grad_accum_steps=2)
print("Best minimal-params run:", best_params)

best_samples = search_min_samples(target_acc=0.91, width=16, depth=2, act_cls=ReLU, lrs=(0.02,0.05,0.1), batch_sizes=(64,128,256), accum_steps=(1,2,4), epochs=250)
print("Best minimal-samples run:", best_samples)


epochs:   7%|▋         | 17/250 [00:00<00:00, 855.00it/s]


Early stopping at epoch 17 (loss 0.533478 vs 0.538802)
{'acc': 0.7944, 'params': 9, 'width': 2, 'depth': 1, 'act': 'ReLU', 'lr': 0.02, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_140019', 'final_loss': 0.5334776103496551}


epochs:   8%|▊         | 19/250 [00:00<00:00, 360.75it/s]


Early stopping at epoch 19 (loss 0.518519 vs 0.523533)
{'acc': 0.7944, 'params': 9, 'width': 2, 'depth': 1, 'act': 'ReLU', 'lr': 0.05, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_140019', 'final_loss': 0.5185189217329025}


epochs:   6%|▌         | 15/250 [00:00<00:00, 692.26it/s]

...

Early stopping at epoch 35 (loss 0.493004 vs 0.496903)
{'acc': 0.7944, 'samples_seen': 90000, 'lr': 0.1, 'batch_size': 256, 'grad_accum': 1, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_140125', 'final_loss': 0.4930042207241058}


epochs:  10%|▉         | 24/250 [00:00<00:00, 468.93it/s]


Early stopping at epoch 24 (loss 0.496856 vs 0.501836)
{'acc': 0.7944, 'samples_seen': 62500, 'lr': 0.1, 'batch_size': 256, 'grad_accum': 2, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_140125', 'final_loss': 0.4968561053276062}


epochs:  14%|█▍        | 35/250 [00:00<00:00, 524.43it/s]


Early stopping at epoch 35 (loss 0.484889 vs 0.489216)
{'acc': 0.7944, 'samples_seen': 90000, 'lr': 0.1, 'batch_size': 256, 'grad_accum': 4, 'run_dir': '/home/karthik/assignment-3-karthikvmala-main/runs/20251013_140126', 'final_loss': 0.4848893254995346}
Best minimal-samples run: None
